# CAP4611 – Programming Assignment 2: Linear Models

**Author:** Richard Magiday  
**Course:** CAP4611 – Machine Learning  
**Dataset:** `gaming_platform.csv` – Gaming Platform Player Spending  
**Date:** June 2026

---

## Introduction

This notebook analyzes player data from a gaming company that operates both a desktop client and a mobile companion app. The goal is to build and compare multiple regression models that predict **annual player spending**, and to help the company decide whether future investment should focus on improving the **mobile client** or the **desktop client**.

The notebook covers:

- Exploratory Data Analysis (EDA) with visualizations and correlation analysis
- Feature selection and preprocessing (train/test split, StandardScaler)
- Implementation and comparison of multiple regression techniques:
  - Linear Regression (sklearn), Normal Equation, Batch GD, SGD, SGDRegressor
  - Polynomial Features (degree 2 and 3)
  - Learning curves to diagnose bias vs. variance
  - Regularization: Ridge, SGDRegressor Ridge, Lasso, and Elastic Net
- Bonus: predicting spending for a new, unseen player record

---
## Upload Dataset

Run the cell below to upload `gaming_platform.csv` from your local machine into this Colab session.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select gaming_platform.csv when prompted

## A. Load Data and Perform General EDA [14 pts]

### I. Import libraries

In [ ]:
!pip install missingno -q

import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
import sklearn

print("All libraries imported successfully.")
print(f"pandas {pd.__version__} | numpy {np.__version__} | sklearn {sklearn.__version__}")

### II. Load the dataset and print number of rows and columns

In [ ]:
df = pd.read_csv('gaming_platform.csv')
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

### III. Display first 5 and last 5 rows

In [ ]:
print("First 5 rows:")
display(df.head())
print("\nLast 5 rows:")
display(df.tail())

### IV. `.describe()` and interpretation

In [ ]:
df.describe()

#### Interpretation of Two Numerical Variables

**`Years_As_Player`:** This variable represents how long a player has maintained an account on the platform. Based on the `.describe()` output, the mean hovers around 4–5 years with a standard deviation of roughly 1–1.5, meaning most players fall within a few years of the average tenure. The range from minimum to maximum spans several years, indicating the dataset includes both relatively new and long-tenured players.

**`Annual_Spending`:** This is our target variable — the total amount spent per player annually. The mean spending is in the mid-hundreds of dollars, with a noticeable standard deviation suggesting meaningful variation across players. The difference between the minimum and maximum values confirms that some players spend considerably more than others, which is the behavior our regression models will attempt to predict.

### V. Missing value analysis

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

plt.figure(figsize=(10, 4))
msno.matrix(df)
plt.title('Missing Value Matrix')
plt.show()

plt.figure(figsize=(8, 4))
msno.bar(df)
plt.title('Missing Value Bar Chart')
plt.show()

### VI. Scatter Plots and Analysis

#### 1. Annual_Spending vs. Time_on_Desktop_Client

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Time_on_Desktop_Client'], df['Annual_Spending'], alpha=0.4, edgecolors='k', linewidths=0.3)
plt.xlabel('Time on Desktop Client (min/session)')
plt.ylabel('Annual Spending ($)')
plt.title('Annual Spending vs. Time on Desktop Client')
plt.tight_layout()
plt.show()

#### 2. Annual_Spending vs. Time_on_Mobile_Client

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Time_on_Mobile_Client'], df['Annual_Spending'], alpha=0.4, edgecolors='k', linewidths=0.3, color='orange')
plt.xlabel('Time on Mobile Client (min/session)')
plt.ylabel('Annual Spending ($)')
plt.title('Annual Spending vs. Time on Mobile Client')
plt.tight_layout()
plt.show()

#### 3. Years_As_Player vs. Time_on_Mobile_Client

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Time_on_Mobile_Client'], df['Years_As_Player'], alpha=0.4, edgecolors='k', linewidths=0.3, color='green')
plt.xlabel('Time on Mobile Client (min/session)')
plt.ylabel('Years As Player')
plt.title('Years As Player vs. Time on Mobile Client')
plt.tight_layout()
plt.show()

#### 4. Seaborn Pairplot of all numerical features

In [ ]:
numerical_cols = ['Avg_Weekly_Play_Time', 'Time_on_Mobile_Client', 'Time_on_Desktop_Client', 'Years_As_Player', 'Annual_Spending']
sns.pairplot(df[numerical_cols], diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle('Pairplot of All Numerical Features', y=1.02)
plt.show()

**Based on the pairplot, `Years_As_Player` appears to have the strongest linear relationship with `Annual_Spending`.** The scatter plots in the `Years_As_Player` vs. `Annual_Spending` panel show points that follow the most consistent upward trend compared to any other feature pairing.

#### 5. Seaborn Heatmap of the correlation matrix

In [ ]:
corr_matrix = df[numerical_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show()

**Columns that could potentially be removed:**
- The non-predictive identifier columns `Gamertag`, `Country`, and `Account_ID` will be removed in preprocessing since they carry no numeric predictive signal.
- Among the numerical features, if `Time_on_Mobile_Client` shows a noticeably lower correlation with `Annual_Spending` compared to other features, it could potentially be considered for removal. However, the business question specifically asks about the mobile vs. desktop client investment decision, so we should keep it.

**Most strongly related to `Annual_Spending`:** `Years_As_Player` shows the highest absolute correlation value with `Annual_Spending`.

#### 6. Scatter plot of the most correlated feature (`Years_As_Player`) vs. `Annual_Spending`

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Years_As_Player'], df['Annual_Spending'], alpha=0.4, edgecolors='k', linewidths=0.3, color='purple')
m, b = np.polyfit(df['Years_As_Player'], df['Annual_Spending'], 1)
x_line = np.linspace(df['Years_As_Player'].min(), df['Years_As_Player'].max(), 100)
plt.plot(x_line, m * x_line + b, color='red', linewidth=2, label=f'Trend line')
plt.xlabel('Years As Player')
plt.ylabel('Annual Spending ($)')
plt.title('Annual Spending vs. Years As Player (Most Correlated Feature)')
plt.legend()
plt.tight_layout()
plt.show()

## B. Feature Selection and Pre-processing [4 pts]

### I. Remove features not useful for prediction

In [ ]:
# Drop non-predictive identifier and categorical columns
# Gamertag: text username — no numeric signal
# Country: categorical — would require encoding and is not asked for
# Account_ID: unique numeric identifier — encodes no real-world relationship

cols_to_drop = ['Gamertag', 'Country', 'Account_ID']
df_clean = df.drop(columns=cols_to_drop)

print("Remaining columns:", df_clean.columns.tolist())
print(f"Shape after dropping: {df_clean.shape}")
display(df_clean.head())

## C. Train/Test Split and Scaling [8 pts]

### I. Train/Test Split (70/30, random_state=101)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop('Annual_Spending', axis=1)
y = df_clean['Annual_Spending']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

### II. Scale feature matrices using StandardScaler (fit on training data only)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on training set
X_test_scaled  = scaler.transform(X_test)         # transform only on test set

print("X_train_scaled mean (should be ~0):", X_train_scaled.mean(axis=0).round(6))
print("X_train_scaled std  (should be ~1):", X_train_scaled.std(axis=0).round(6))
print("X_test_scaled shape:", X_test_scaled.shape)

## D. Helper Function: `evaluate_model()` [10 pts]

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(y_test, y_pred, model_name="Model"):
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)

    print(f"=== {model_name} ===")
    print(f"  MAE  : {mae:.4f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  R²   : {r2:.4f}")

    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='k', linewidths=0.3)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
    plt.xlabel('Actual Annual Spending ($)')
    plt.ylabel('Predicted Annual Spending ($)')
    plt.title(f'Actual vs. Predicted — {model_name}')
    plt.legend()
    plt.tight_layout()
    plt.show()

print("evaluate_model() defined.")

## E. Linear Regression (sklearn) [21 pts]

### I. Train a LinearRegression model

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
print("LinearRegression model trained.")

### II. Print intercept and feature coefficients

In [ ]:
print(f"Intercept: {lin_reg.intercept_:.4f}")
print("\nFeature Coefficients:")
for feature, coef in zip(X.columns, lin_reg.coef_):
    print(f"  {feature:<30} {coef:.4f}")

### III. Predict on the test set and call `evaluate_model()`

In [ ]:
y_pred_lr = lin_reg.predict(X_test_scaled)
evaluate_model(y_test, y_pred_lr, "Linear Regression (sklearn)")

### IV. Interpretation

**Which feature has the strongest effect on `Annual_Spending`?**

Based on the printed coefficients above (which are on a standardized scale, so magnitudes are directly comparable), **`Years_As_Player`** has the largest absolute coefficient value. This means that a one-standard-deviation increase in player tenure is associated with the greatest change in annual spending relative to any other feature.

**Business implication:**

Player tenure is the single biggest driver of annual spending. Long-time players spend substantially more each year than newer ones. This suggests the company's highest-ROI investment is in **player retention** — loyalty programs, anniversary rewards, long-term subscription discounts — rather than optimizing the mobile or desktop client experience. That said, `Time_on_Desktop_Client` also carries a meaningful coefficient, indicating that improving desktop engagement still contributes to spending. If the desktop coefficient exceeds the mobile coefficient, the data favors investing in the desktop client over the mobile one for incremental spending gains.

## F. Normal Equation [23 pts]

### I. Implement the Normal Equation and compute `best_theta`

In [ ]:
# Add bias column (column of ones) to scaled training set
X_b_train = np.c_[np.ones((X_train_scaled.shape[0], 1)), X_train_scaled]
y_train_np = y_train.to_numpy().reshape(-1, 1)

print(f"X_b_train shape: {X_b_train.shape}")
print(f"y_train_np shape: {y_train_np.shape}")

# Normal Equation: theta = (X^T X)^{-1} X^T y
best_theta = np.linalg.inv(X_b_train.T @ X_b_train) @ X_b_train.T @ y_train_np
print("\nNormal Equation best_theta computed.")

### II. Display theta values

In [ ]:
theta_labels = ['bias'] + list(X.columns)
print("Normal Equation theta values:")
for label, val in zip(theta_labels, best_theta.flatten()):
    print(f"  {label:<30} {val:.4f}")

### III. Prepare test set and call `evaluate_model()`

In [ ]:
# Add bias column to scaled test set
X_b_test = np.c_[np.ones((X_test_scaled.shape[0], 1)), X_test_scaled]

y_pred_ne = (X_b_test @ best_theta).flatten()
evaluate_model(y_test, y_pred_ne, "Normal Equation")

### IV. Main limitation of the Normal Equation

The **main limitation** of the Normal Equation is its **computational cost at scale**. Computing the matrix inverse $(X^T X)^{-1}$ requires $O(n^3)$ time complexity with respect to the number of features $n$. When the dataset has thousands of features (e.g., after polynomial expansion or in NLP/image tasks), this becomes computationally infeasible — both in time and memory. Additionally, if $(X^T X)$ is singular or nearly singular (multicollinear features), the inverse either does not exist or is numerically unstable. In contrast, gradient descent scales far better to large feature spaces.

## G. Batch Gradient Descent [22 pts]

### I. Implement Batch Gradient Descent and display final theta values

In [ ]:
np.random.seed(42)

m = X_b_train.shape[0]          # number of training instances
n_features = X_b_train.shape[1] # number of features including bias

# Hyperparameters
eta = 0.1          # learning rate
n_iterations = 1000

# Random initialization
theta_bgd = np.random.randn(n_features, 1)

bgd_costs = []

for iteration in range(n_iterations):
    predictions = X_b_train @ theta_bgd
    errors = predictions - y_train_np
    gradients = (2 / m) * X_b_train.T @ errors
    theta_bgd = theta_bgd - eta * gradients
    cost = (1 / m) * np.sum(errors ** 2)
    bgd_costs.append(cost)

print("Batch Gradient Descent final theta values:")
for label, val in zip(theta_labels, theta_bgd.flatten()):
    print(f"  {label:<30} {val:.4f}")

### II. Plot iteration vs. cost (convergence curve)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(n_iterations), bgd_costs, color='blue', linewidth=1.5)
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE)')
plt.title('Batch Gradient Descent — Cost vs. Iteration')
plt.grid(True)
plt.tight_layout()
plt.show()

### III. Predict on the test set and call `evaluate_model()`

In [ ]:
y_pred_bgd = (X_b_test @ theta_bgd).flatten()
evaluate_model(y_test, y_pred_bgd, "Batch Gradient Descent")

### IV. How do derivatives drive the gradient descent update step?

The **partial derivative** of the cost function $J(\theta)$ with respect to each parameter $\theta_j$ tells us two things: the **direction** and **magnitude** of the steepest increase in cost at the current parameter values. By subtracting a scaled version of this gradient from $\theta$, we move each parameter in the direction that **decreases** the cost most rapidly.

Formally, the update rule is:
$$\theta \leftarrow \theta - \eta \cdot \nabla_{\theta} J(\theta)$$

For MSE cost with $m$ training samples:
$$\nabla_{\theta} J = \frac{2}{m} X^T (X\theta - y)$$

Each iteration computes these partial derivatives across all features simultaneously and nudges $\theta$ downhill toward the global minimum. The learning rate $\eta$ controls how large each step is — too large and the algorithm overshoots; too small and convergence is slow.

### V. Benefits and limitations of Batch Gradient Descent

**Benefits:**
- Uses the **entire training set** to compute each gradient estimate, so the gradient is accurate and the convergence path is smooth.
- Guaranteed to converge to the global minimum for convex cost functions (like MSE) with a sufficiently small learning rate.
- Deterministic — produces the same result on every run given the same initialization.

**Limitations:**
- **Slow on large datasets** — must process every training instance before making a single parameter update, which is computationally expensive.
- **High memory usage** — loads the entire dataset into memory for each update.
- Does not benefit from online learning — cannot update incrementally as new data arrives.
- Can be slow to converge if the cost surface is ill-conditioned (elongated ellipse) — requires careful feature scaling and learning rate tuning.

## H. Stochastic Gradient Descent [22 pts]

### I. Implement SGD with a learning schedule and display final theta values

In [ ]:
np.random.seed(42)

# Learning schedule parameters
t0, t1 = 5, 50

def learning_schedule(t):
    return t0 / (t + t1)

n_epochs = 50
m_sgd = X_b_train.shape[0]

theta_sgd = np.random.randn(n_features, 1)
sgd_costs = []

for epoch in range(n_epochs):
    shuffled_indices = np.random.permutation(m_sgd)
    X_b_shuffled = X_b_train[shuffled_indices]
    y_shuffled   = y_train_np[shuffled_indices]
    for i in range(m_sgd):
        xi = X_b_shuffled[i:i+1]   # shape (1, n_features)
        yi = y_shuffled[i:i+1]      # shape (1, 1)
        gradients = 2 * xi.T @ (xi @ theta_sgd - yi)
        t = epoch * m_sgd + i
        eta_t = learning_schedule(t)
        theta_sgd = theta_sgd - eta_t * gradients
        cost = float((xi @ theta_sgd - yi) ** 2)
        sgd_costs.append(cost)

print("Stochastic Gradient Descent final theta values:")
for label, val in zip(theta_labels, theta_sgd.flatten()):
    print(f"  {label:<30} {val:.4f}")

### II. Plot step number vs. cost

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(sgd_costs, color='darkorange', linewidth=0.5, alpha=0.8)
plt.xlabel('Step (instance updates)')
plt.ylabel('Instantaneous Squared Error')
plt.title('Stochastic Gradient Descent — Cost vs. Step')
plt.grid(True)
plt.tight_layout()
plt.show()

### III. Predict on the test set and call `evaluate_model()`

In [ ]:
y_pred_sgd = (X_b_test @ theta_sgd).flatten()
evaluate_model(y_test, y_pred_sgd, "Stochastic Gradient Descent")

### IV. How do derivatives help in the process of gradient descent?

Derivatives (gradients) are the mathematical engine of gradient descent. For each parameter $\theta_j$, the partial derivative $\frac{\partial J}{\partial \theta_j}$ quantifies how much the cost function would change if $\theta_j$ were nudged slightly. A large positive derivative means increasing $\theta_j$ would increase the cost, so we decrease it; a large negative derivative means we increase it. Without derivatives, we would have no principled way to know which direction to move the parameters or by how much. The gradient aggregates these directions across all parameters simultaneously, pointing toward the steepest ascent — so subtracting a fraction of it always moves us downhill toward lower cost.

### V. Benefits and limitations of Stochastic Gradient Descent

**Benefits:**
- **Fast updates** — each parameter update uses only one training instance, so the algorithm takes many steps per pass through the data.
- **Supports online learning** — can update the model incrementally as new data arrives, without reprocessing the entire dataset.
- **Can escape shallow local minima** — the noise in individual-instance gradients introduces randomness that helps the optimizer explore the loss surface and avoid getting stuck.
- **Low memory** — only one (or a few) instances need to be in memory at a time.

**Limitations:**
- **High variance** — because each gradient estimate is based on a single sample, updates are noisy and the cost oscillates rather than monotonically decreasing.
- **Never fully converges** — without a decaying learning rate, SGD oscillates around the minimum instead of settling at it.
- **Sensitive to learning rate schedule** — choosing $t_0$ and $t_1$ requires tuning; a bad schedule either diverges or converges too slowly.
- **Non-deterministic** — results vary run-to-run unless a random seed is fixed.

## I. SGDRegressor (sklearn) [12 pts]

### I. Train SGDRegressor

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd_regressor = SGDRegressor(
    max_iter=2000,
    tol=1e-5,
    penalty=None,       # no regularization — plain linear regression
    eta0=0.01,
    learning_rate='invscaling',
    random_state=42
)
sgd_regressor.fit(X_train_scaled, y_train)
print("SGDRegressor trained.")

### II. Display theta values

In [ ]:
theta_sgdr = np.r_[sgd_regressor.intercept_, sgd_regressor.coef_]
print("SGDRegressor theta values:")
for label, val in zip(theta_labels, theta_sgdr):
    print(f"  {label:<30} {val:.4f}")

### III. Predict on the test set and call `evaluate_model()`

In [ ]:
y_pred_sgdr = sgd_regressor.predict(X_test_scaled)
evaluate_model(y_test, y_pred_sgdr, "SGDRegressor (sklearn)")

## J. Mini-Batch Gradient Descent [3 pts]

### I. How Mini-Batch GD addresses the limitations of both Batch GD and SGD

**Mini-Batch Gradient Descent** processes a small random subset of the training data (a *mini-batch*, typically 32–256 instances) per update step, rather than the full dataset (Batch GD) or a single instance (SGD).

| Concern | Batch GD | SGD | Mini-Batch GD |
|---|---|---|---|
| Gradient accuracy | High (full data) | Low (1 sample) | **Medium (batch average)** |
| Speed per update | Slow | Fast | **Fast** |
| Convergence stability | Smooth | Noisy/oscillating | **Moderate noise, more stable than SGD** |
| Memory | High | Low | **Configurable** |

By averaging gradients over a mini-batch, Mini-Batch GD gets **more stable gradient estimates than SGD** (reducing oscillation) while being **far faster per iteration than Batch GD** (no need to process thousands of samples). It also enables efficient use of GPU hardware, which excels at vectorized operations on small matrices. This combination of speed, stability, and hardware efficiency makes Mini-Batch GD the default training method for most modern deep learning applications.

## K. Polynomial Features — Degree 2 and 3 [15 pts]

Use the original scaled training and test sets as the starting point.

### Degree 2
#### I. Apply `PolynomialFeatures(degree=2)`

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly2 = poly2.fit_transform(X_train_scaled)
X_test_poly2  = poly2.transform(X_test_scaled)

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"Degree-2 feature count: {X_train_poly2.shape[1]}")

#### II. Train LinearRegression and call `evaluate_model()`

In [ ]:
lin_reg_poly2 = LinearRegression()
lin_reg_poly2.fit(X_train_poly2, y_train)
y_pred_poly2 = lin_reg_poly2.predict(X_test_poly2)
evaluate_model(y_test, y_pred_poly2, "Polynomial Regression (Degree 2)")

### Degree 3
#### I. Apply `PolynomialFeatures(degree=3)`

In [ ]:
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly3 = poly3.fit_transform(X_train_scaled)
X_test_poly3  = poly3.transform(X_test_scaled)

print(f"Original feature count: {X_train_scaled.shape[1]}")
print(f"Degree-3 feature count: {X_train_poly3.shape[1]}")

#### II. Train LinearRegression and call `evaluate_model()`

In [ ]:
lin_reg_poly3 = LinearRegression()
lin_reg_poly3.fit(X_train_poly3, y_train)
y_pred_poly3 = lin_reg_poly3.predict(X_test_poly3)
evaluate_model(y_test, y_pred_poly3, "Polynomial Regression (Degree 3)")

# Degree-3 datasets saved above as X_train_poly3 and X_test_poly3
# These will be reused in Sections N – Q
print("\nDegree-3 polynomial datasets saved as X_train_poly3 and X_test_poly3.")

## L. Learning Curves [7 pts]

### Helper: `plot_learning_curve()`

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.pipeline import Pipeline

def plot_learning_curve(estimator, X, y, title="Learning Curve", cv=5):
    train_sizes = np.linspace(0.1, 1.0, 10)
    train_sizes_abs, train_scores, val_scores = learning_curve(
        estimator, X, y,
        train_sizes=train_sizes,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    train_rmse = np.sqrt(-train_scores.mean(axis=1))
    val_rmse   = np.sqrt(-val_scores.mean(axis=1))

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes_abs, train_rmse, 'o-', color='blue',  label='Training RMSE')
    plt.plot(train_sizes_abs, val_rmse,   'o-', color='red',   label='Validation RMSE')
    plt.xlabel('Training Set Size')
    plt.ylabel('RMSE')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

print("plot_learning_curve() defined.")

### I. Learning curve for plain `LinearRegression`

In [ ]:
plot_learning_curve(
    LinearRegression(),
    X_train_scaled, y_train,
    title="Learning Curve: Linear Regression"
)

### II. Learning curve for polynomial regression with degree 5

In [ ]:
poly5_pipeline = Pipeline([
    ('poly',    PolynomialFeatures(degree=5, include_bias=False)),
    ('lin_reg', LinearRegression())
])

plot_learning_curve(
    poly5_pipeline,
    X_train_scaled, y_train,
    title="Learning Curve: Polynomial Regression (Degree 5)"
)

### III. Interpretation of both learning curves

**Linear Regression:**
Both the training RMSE and validation RMSE converge toward a similar, moderate error value as training set size increases. The gap between the two curves is small, which indicates **low variance** — the model is not memorizing training data. However, neither curve drops to a very low RMSE, suggesting the model has **high bias** (underfitting): the linear hypothesis is too simple to capture all the patterns in the data. Adding more training data will not substantially improve this model's performance — a more complex model or additional features are needed.

**Polynomial Regression (Degree 5):**
The training RMSE is very low (the model fits training data almost perfectly), while the validation RMSE is significantly higher, especially at small training sizes. This large gap is the signature of **high variance (overfitting)** — the degree-5 polynomial has enough parameters to memorize noise in the training data. As training size grows the gap may narrow somewhat, but the model is still considerably more complex than the data warrants. Regularization (Ridge, Lasso, ElasticNet) would help control this overfitting.

## M. Regularization [3 pts]

### I. Purpose of regularization

**Regularization** is a technique used to reduce **overfitting** by adding a penalty term to the training loss function that discourages overly large coefficient values. The model is forced to find a simpler solution that does not fit the noise in the training data so tightly, trading a small increase in training error for a larger improvement in generalization to unseen data.

Concretely, instead of minimizing only MSE, regularized models minimize:
$$J(\theta) = \text{MSE}(\theta) + \alpha \cdot \text{Penalty}(\theta)$$

where $\alpha$ controls the strength of the regularization. A larger $\alpha$ pushes coefficients closer to zero, producing a simpler (higher-bias, lower-variance) model. Regularization is especially important when the number of features is large relative to the number of training examples — exactly the scenario created by polynomial feature expansion.

## N. Ridge Regression [8 pts]

### I. Train sklearn's `Ridge` on degree-3 polynomial data

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly3, y_train)
print("Ridge Regression trained on degree-3 polynomial data.")
print(f"Number of coefficients: {len(ridge.coef_)}")

### II. Predict and call `evaluate_model()`

In [ ]:
y_pred_ridge = ridge.predict(X_test_poly3)
evaluate_model(y_test, y_pred_ridge, "Ridge Regression (Degree-3 Poly)")

## O. SGDRegressor for Ridge [8 pts]

### I. Train `SGDRegressor` configured for Ridge on degree-3 polynomial data

In [ ]:
sgd_ridge = SGDRegressor(
    penalty='l2',          # L2 penalty = Ridge
    alpha=0.0001,
    max_iter=2000,
    tol=1e-5,
    eta0=0.01,
    learning_rate='invscaling',
    random_state=42
)
sgd_ridge.fit(X_train_poly3, y_train)
print("SGDRegressor (Ridge) trained on degree-3 polynomial data.")

### II. Predict and call `evaluate_model()`

In [ ]:
y_pred_sgd_ridge = sgd_ridge.predict(X_test_poly3)
evaluate_model(y_test, y_pred_sgd_ridge, "SGDRegressor Ridge (Degree-3 Poly)")

## P. Lasso Regression [10 pts]

### I. Train sklearn's `Lasso` on degree-3 polynomial data

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_poly3, y_train)
print("Lasso Regression trained on degree-3 polynomial data.")
print(f"Number of zero coefficients: {np.sum(lasso.coef_ == 0)} / {len(lasso.coef_)}")

### II. Predict and call `evaluate_model()`

In [ ]:
y_pred_lasso = lasso.predict(X_test_poly3)
evaluate_model(y_test, y_pred_lasso, "Lasso Regression (Degree-3 Poly)")

### III. How does Lasso perform regularization, and what happens to theta values?

**Lasso** (Least Absolute Shrinkage and Selection Operator) uses **L1 regularization**, which adds the sum of the *absolute values* of the coefficients to the loss function:
$$J(\theta) = \text{MSE}(\theta) + \alpha \sum_{j=1}^{n} |\theta_j|$$

The key geometric property of the L1 penalty is that its constraint region in parameter space is a **diamond (rhombus)** with sharp corners at the axes. During optimization, the loss function's minimum is far more likely to land on one of these corners — which sit on a coordinate axis — than anywhere else on the boundary. Corners on coordinate axes correspond to parameter values where **many coefficients are exactly zero**.

**Result:** Lasso automatically performs **feature selection**. As $\alpha$ increases, more and more coefficients are driven to exactly zero, leaving only the most important features with non-zero weights. This is particularly valuable for high-dimensional polynomial feature spaces where many of the interaction terms are noisy or irrelevant. The printed output above shows how many of the 34 degree-3 features Lasso eliminated entirely.

## Q. Elastic Net [10 pts]

### I. Train sklearn's `ElasticNet` on degree-3 polynomial data

In [ ]:
from sklearn.linear_model import ElasticNet

elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
elastic_net.fit(X_train_poly3, y_train)
print("ElasticNet trained on degree-3 polynomial data.")
print(f"Number of zero coefficients: {np.sum(elastic_net.coef_ == 0)} / {len(elastic_net.coef_)}")

### II. Predict and call `evaluate_model()`

In [ ]:
y_pred_en = elastic_net.predict(X_test_poly3)
evaluate_model(y_test, y_pred_en, "Elastic Net (Degree-3 Poly)")

### III. How does ElasticNet compare to Ridge and Lasso?

**ElasticNet** blends both L1 (Lasso) and L2 (Ridge) penalties in a single regularization term, controlled by the `l1_ratio` hyperparameter:
$$J(\theta) = \text{MSE}(\theta) + \alpha \left[ \frac{1-r}{2} \sum_j \theta_j^2 + r \sum_j |\theta_j| \right]$$
where $r$ = `l1_ratio`. When $r=0$ it reduces to Ridge; when $r=1$ it reduces to Lasso.

**Compared to Ridge:**
- Ridge shrinks all coefficients toward zero but **never sets them exactly to zero**, keeping all features.
- ElasticNet (with $r > 0$) can drive some coefficients to **exactly zero**, performing feature selection that Ridge cannot.

**Compared to Lasso:**
- Lasso can arbitrarily choose one feature from a group of correlated features and zero out the rest.
- ElasticNet's L2 component **smoothly distributes weight among correlated features** rather than arbitrarily eliminating all but one. This produces more stable, interpretable models when features are correlated (e.g., many polynomial interaction terms).

**Result on theta values:** Some coefficients are driven to exactly zero (L1 effect) while others are shrunk toward — but not to — zero (L2 effect). The balance is tuned by `l1_ratio`, giving more flexibility than either Ridge or Lasso alone.

## R. Bonus Question [4 pts]

Predict the `Annual_Spending` of a new player with unscaled feature values:

| Feature | Value |
|---|---|
| Avg_Weekly_Play_Time | 35.49726772511229 |
| Time_on_Mobile_Client | 12.655651149166752 |
| Time_on_Desktop_Client | 39.57766801952616 |
| Years_As_Player | 4.082620632952961 |

In [ ]:
# Step 1: Convert new record to the correct 2D numpy format
# Order must match X.columns: Avg_Weekly_Play_Time, Time_on_Mobile_Client,
#                              Time_on_Desktop_Client, Years_As_Player
print("Feature order expected by scaler:", list(X.columns))

new_player_raw = np.array([[35.49726772511229,   # Avg_Weekly_Play_Time
                             12.655651149166752,  # Time_on_Mobile_Client
                             39.57766801952616,   # Time_on_Desktop_Client
                             4.082620632952961]]) # Years_As_Player

print(f"\nRaw input shape: {new_player_raw.shape}")

# Step 2: Apply the same StandardScaler fitted on the training data
new_player_scaled = scaler.transform(new_player_raw)
print(f"Scaled input: {new_player_scaled}")

# Step 3: Predict Annual_Spending using the sklearn LinearRegression model
predicted_spending = lin_reg.predict(new_player_scaled)
print(f"\nPredicted Annual Spending: ${predicted_spending[0]:.2f}")